In [2]:
import os
import json
import faiss
from sentence_transformers import SentenceTransformer

In [3]:
# Config

KNOWLEDGE_DIR = "RAG_knowledge/original"
OUTPUT_DIR = "RAG_knowledge/index"
CHUNK_SIZE = 250
CHUNK_OVERLAP = 50

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
# Utility Functions

def load_markdown(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()


def chunk_text(text, chunk_size, overlap):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [5]:
# Indexing Logic

def index_markdown_file(md_path):
    file_name = os.path.splitext(os.path.basename(md_path))[0]
    print(f"Indexing: {file_name}")

    text = load_markdown(md_path)
    chunks = chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)

    embeddings = embedding_model.encode(chunks)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)

    # Create output folder
    file_output_dir = os.path.join(OUTPUT_DIR, file_name)
    os.makedirs(file_output_dir, exist_ok=True)

    # Save FAISS index
    faiss.write_index(index, os.path.join(file_output_dir, "index.faiss"))

    # Save chunks + metadata
    metadata = {
        "source_file": md_path,
        "num_chunks": len(chunks),
        "chunks": chunks
    }

    with open(os.path.join(file_output_dir, "chunks.json"), "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print(f"Saved index to {file_output_dir}\n")



In [6]:
# Walk through all MD files

for root, _, files in os.walk(KNOWLEDGE_DIR):
    for file in files:
        if file.endswith(".md"):
            full_path = os.path.join(root, file)
            index_markdown_file(full_path)

print("Indexing complete.")

Indexing: fast_speaking_rate
Saved index to RAG_knowledge/index\fast_speaking_rate

Indexing: filler_words
Saved index to RAG_knowledge/index\filler_words

Indexing: stuttering
Saved index to RAG_knowledge/index\stuttering

Indexing: nervouseness
Saved index to RAG_knowledge/index\nervouseness

Indexing complete.
